In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.gold.race_strategy_performance"



In [0]:
# 1. Load Silver Tables
df_results = spark.table(f"{catalog_name}.silver.fact_results")
df_drivers = spark.table(f"{catalog_name}.silver.dim_drivers")
df_constructors = spark.table(f"{catalog_name}.silver.dim_constructors")
df_races = spark.table(f"{catalog_name}.silver.dim_races")
df_circuits = spark.table(f"{catalog_name}.silver.dim_circuits")
df_pit_stops = spark.table(f"{catalog_name}.silver.fact_pit_stops")
df_lap_times = spark.table(f"{catalog_name}.silver.fact_lap_times")



In [0]:
# 2. Aggregations on Pit Stops
df_pit_agg = (
    df_pit_stops
    .groupBy("race_id", "driver_id")
    .agg(
        F.count("stop_number").alias("total_pit_stops"),
        F.round(F.avg("duration_ms") / 1000, 2).alias("avg_pit_duration_sec"),
        F.round(F.min("duration_ms") / 1000, 2).alias("fastest_pit_duration_sec")
    )
)



In [0]:
# 3. Aggregations on Lap Pace
df_lap_agg = (
    df_lap_times
    .groupBy("race_id", "driver_id")
    .agg(
        F.round(F.avg("lap_time_ms") / 1000, 2).alias("avg_lap_time_sec"),
        F.round(F.stddev("lap_time_ms") / 1000, 2).alias("lap_time_stddev_sec") # Pace consistency
    )
)



In [0]:
# 4. Join Facts and Dimensions into Gold Product
df_gold = (
    df_results
    .join(df_races, "race_id", "inner")
    .join(df_circuits, "circuit_id", "inner")
    .join(df_drivers, "driver_id", "inner")
    .join(df_constructors, "constructor_id", "inner")
    .join(df_pit_agg, ["race_id", "driver_id"], "left")
    .join(df_lap_agg, ["race_id", "driver_id"], "left")
    .select(
        df_races["race_year"],
        df_races["race_round"],
        df_races["race_name"],
        df_circuits["circuit_name"],
        df_constructors["constructor_name"],
        df_drivers["driver_full_name"],
        df_drivers["anonymized_driver_ref"],
        df_results["grid_position"],
        df_results["position"].alias("finish_position"),
        (df_results["grid_position"] - df_results["position"]).alias("positions_gained_lost"),
        df_results["points"],
        F.coalesce(df_pit_agg["total_pit_stops"], F.lit(0)).alias("total_pit_stops"),
        df_pit_agg["avg_pit_duration_sec"],
        df_pit_agg["fastest_pit_duration_sec"],
        df_lap_agg["avg_lap_time_sec"],
        df_lap_agg["lap_time_stddev_sec"]
    )
)



In [0]:
# 5. Save Data Product to Gold Schema with schema overwrite enabled
(
    df_gold.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)


In [0]:
# 6. Apply Business Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'gold',
        'data_product' = 'race_strategy_analytics',
        'domain' = 'performance_engineering',
        'contains_pii' = 'false'
    )
""")

print(f"✓ Data product successfully created and governed: {target_table}")